# SchoolBridge Fine-Tuning with Unsloth

Fine-tune Gemma 4 E4B for school notice extraction using QLoRA.

**Requirements**: Run on Kaggle with GPU T4 x2 or P100 accelerator enabled.

In [ ]:
%%capture
!pip install unsloth
!pip install --upgrade --no-cache-dir --no-deps unsloth unsloth_zoo
!pip install --upgrade transformers datasets trl accelerate peft bitsandbytes sentencepiece protobuf

In [ ]:
from unsloth import FastModel
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name="unsloth/gemma-4-E4B-it",
    dtype=None,
    max_seq_length=2048,
    load_in_4bit=True,
    full_finetuning=False,
)

print("Model loaded successfully!")
print(f"Model type: {type(model).__name__}")

In [ ]:
model = FastModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

model.print_trainable_parameters()

In [ ]:
import os, glob
from datasets import load_dataset

# Try standard path first, then discover dataset location
data_dir = "/kaggle/input/schoolbridge-training-data"
train_file = os.path.join(data_dir, "train.jsonl")

if not os.path.exists(train_file):
    print(f"Dataset not found at {data_dir}, searching...")
    # List all input directories
    input_base = "/kaggle/input"
    if os.path.exists(input_base):
        for d in os.listdir(input_base):
            full = os.path.join(input_base, d)
            print(f"  Found input: {full}")
            for f in os.listdir(full):
                print(f"    - {f}")
            if os.path.exists(os.path.join(full, "train.jsonl")):
                data_dir = full
                train_file = os.path.join(full, "train.jsonl")
                print(f"  -> Using: {data_dir}")
                break
    else:
        print(f"  {input_base} does not exist!")

if not os.path.exists(train_file):
    print("Dataset not mounted! Downloading from Kaggle...")
    os.system("pip install -q kaggle")
    os.makedirs("/kaggle/input/schoolbridge-training-data", exist_ok=True)
    os.system("kaggle datasets download rohanpatnaik/schoolbridge-training-data -p /tmp/sb-data --unzip")
    os.system("cp /tmp/sb-data/*.jsonl /kaggle/input/schoolbridge-training-data/")
    data_dir = "/kaggle/input/schoolbridge-training-data"
    train_file = os.path.join(data_dir, "train.jsonl")

val_file = os.path.join(data_dir, "val.jsonl")
print(f"Train file: {train_file} (exists: {os.path.exists(train_file)})")
print(f"Val file: {val_file} (exists: {os.path.exists(val_file)})")

dataset = load_dataset("json", data_files={
    "train": train_file,
    "validation": val_file,
})

print(f"Train: {len(dataset['train'])} examples")
print(f"Validation: {len(dataset['validation'])} examples")
print(f"\nSample keys: {list(dataset['train'][0].keys())}")
print(f"Sample conversation roles: {[c['role'] for c in dataset['train'][0]['conversations']]}")

In [ ]:
# Format conversations using the tokenizer's built-in chat template
def apply_template(examples):
    texts = []
    for convo in examples["conversations"]:
        # Convert to the format expected by apply_chat_template
        messages = []
        for turn in convo:
            messages.append({"role": turn["role"], "content": turn["content"]})
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(apply_template, batched=True)

print("Sample formatted text (first 600 chars):")
print(dataset["train"][0]["text"][:600])
print("\n...")
print(f"\nTotal length: {len(dataset['train'][0]['text'])} chars")

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    args=TrainingArguments(
        output_dir="./outputs",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        weight_decay=0.01,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=100,
        save_strategy="no",
        seed=42,
        report_to="none",
    ),
    dataset_text_field="text",
    max_seq_length=2048,
)

# Only train on model responses, not user prompts
trainer = train_on_responses_only(
    trainer,
    instruction_part="<start_of_turn>user\n",
    response_part="<start_of_turn>model\n",
)

print("Trainer ready!")

In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu_stats.name} ({gpu_stats.total_mem / 1024**3:.1f} GB)")
print(f"CUDA: {torch.version.cuda}")
print(f"\nStarting training...\n")

stats = trainer.train()

print(f"\n{'='*50}")
print(f"Training complete!")
print(f"Final train loss: {stats.training_loss:.4f}")
print(f"Runtime: {stats.metrics['train_runtime']:.0f}s ({stats.metrics['train_runtime']/60:.1f} min)")

In [ ]:
# Quick test inference
from transformers import TextStreamer

test_notice = """Dear Parent, Your child has 8 absences this semester exceeding the 5 absence threshold.
Please contact the attendance office within 5 days. State law requires reporting excessive absences.
If medical, provide healthcare documentation. - Lincoln Elementary"""

messages = [
    {"role": "user", "content": f"Analyze this school notice:\n\n{test_notice}\n\nTarget language: English"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to("cuda")

print("Model output:")
_ = model.generate(
    **inputs,
    max_new_tokens=1024,
    temperature=0.1,
    do_sample=True,
    streamer=TextStreamer(tokenizer, skip_prompt=True),
)

In [ ]:
# Save and export to GGUF for Ollama
model.save_pretrained_gguf(
    "schoolbridge-gemma4-e4b",
    tokenizer,
    quantization_method="q4_k_m",
)

print("\nGGUF export complete!")
print("\nNext steps:")
print("1. Download the .gguf file from the output tab")
print("2. Create an Ollama Modelfile pointing to it")
print("3. Run: ollama create schoolbridge -f Modelfile")

import glob
for f in glob.glob("schoolbridge-gemma4-e4b/*"):
    import os
    size_mb = os.path.getsize(f) / 1024 / 1024
    print(f"  {f} ({size_mb:.0f} MB)")